<a href="https://colab.research.google.com/github/tnghks023/ml-study/blob/master/Logistic_Regression_vs_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# 1. 라이브러리 불러오기
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import time

In [12]:
# 2. 데이터 읽기
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
# 3. 결측치 처리
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin은 결측치가 너무 많아서 제거
df = df.drop(columns=["Cabin"])

df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [14]:
# 4. Feature / Target 설정
features = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "SibSp",
    "Parch"
]

X = df[features].copy()
y = df["Survived"]

# Sex 문자 → 숫자 변환
X["Sex"] = X["Sex"].map({
    "male": 0,
    "female": 1
})

X.head()

,Pclass,Sex,Age,Fare,SibSp,Parch
0,3,0,22.0,7.2500,1,0
1,1,1,38.0,71.2833,1,0
2,3,1,26.0,7.9250,0,0
3,1,1,35.0,53.1000,1,0
4,3,0,35.0,8.0500,0,0


In [16]:
# 5. Train / Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
# 6. Logistic Regression은 스케일링 적용
scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000)

log_start = time.perf_counter()

log_model.fit(X_train_scaled, y_train)

log_end = time.perf_counter()

pred_log = log_model.predict(X_test_scaled)

In [18]:
# 7. Random Forest 학습
# Random Forest는 스케일링이 거의 필요 없음
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_start = time.perf_counter()

rf_model.fit(X_train, y_train)

rf_end = time.perf_counter()

pred_rf = rf_model.predict(X_test)

In [19]:
# 8. Logistic Regression 평가
print("Logistic Regression Accuracy:")
print(accuracy_score(y_test, pred_log))

print("\nLogistic Regression Report:")
print(classification_report(y_test, pred_log))

Logistic Regression Accuracy:
0.7988826815642458

Logistic Regression Report:
              precision    recall  f1-score   support

           0       0.81      0.86      0.83       105
           1       0.78      0.72      0.75        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [20]:
# 9. Random Forest 평가
print("Random Forest Accuracy:")
print(accuracy_score(y_test, pred_rf))

print("\nRandom Forest Report:")
print(classification_report(y_test, pred_rf))

Random Forest Accuracy:
0.7988826815642458

Random Forest Report:
              precision    recall  f1-score   support

           0       0.82      0.84      0.83       105
           1       0.76      0.74      0.75        74

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [22]:
print(f"Logistic : {log_end - log_start:.6f} sec")

print(f"Random Forest : {rf_end - rf_start:.6f} sec")

Logistic : 0.004841 sec
Random Forest : 0.212534 sec


* Logistic Regression은 생존이라고 예측한 것의 정확도(Precision) 가 조금 더 높고,
* Random Forest는 실제 생존자를 조금 더 많이 찾아냄(Recall).


학습 속도 차이


Logistic Regression > Random Forest

Random Forest 언제쓰나

* Feature가 수십~수백 개이고 관계가 복잡하면,Random Forest가 Logistic Regression보다 훨씬 좋은 성능을 내는 경우가 많음